# Habitat Gym API Tutorial
Wrap Habitat tasks with the standard Gym interface. Uses datasets from notebook 10.

## Why Gym?

Gym is the **standard RL environment interface** — same API (`reset`, `step`, `observation_space`, `action_space`) used by Stable Baselines, RLlib, etc.

Habitat has **two ways** to expose this:
1. Pre-registered tasks like `HabitatPick-v0` (need rearrange datasets — big downloads)
2. `GymHabitatEnv` — wrap **any** habitat config as a Gym env

This tutorial uses option 2 with the PointNav test scene you already have from notebook 10.

In [36]:
%%bash
# Reuse test scene + pointnav dataset from notebook 10
mkdir -p /content/data/datasets/pointnav /content/data/scene_datasets

ln -sfn /content/drive/MyDrive/HabitatData/versioned_data/habitat_test_pointnav_dataset_1.0 \
    /content/data/datasets/pointnav/habitat-test-scenes

ln -sfn /content/drive/MyDrive/HabitatData/versioned_data/habitat_test_scenes \
    /content/data/scene_datasets/habitat-test-scenes

cat > /tmp/nvidia_egl.json << 'EOF'
{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}
EOF

ls /content/data/datasets/pointnav/habitat-test-scenes/v1/train/train.json.gz && echo "Ready"

/content/data/datasets/pointnav/habitat-test-scenes/v1/train/train.json.gz
Ready


## 2. Compare: Plain Habitat Env vs Gym-wrapped Env

**Plain `habitat.Env`:**
- `obs = env.reset()` → just observations
- `obs = env.step({"action": "move_forward"})` → just observations
- Need to call `env.get_metrics()` separately for rewards/info

**Gym-wrapped env:**
- `obs = env.reset()` → observations
- `obs, reward, done, info = env.step(action)` → 4-tuple standard RL interface
- Standardized `observation_space` and `action_space` properties

## 3. Plain habitat.Env — basic step loop

In [37]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv
cd /content && export MPLBACKEND=Agg
export __EGL_VENDOR_LIBRARY_FILENAMES=/tmp/nvidia_egl.json
export LD_LIBRARY_PATH=$(dirname $(find /usr -name "libEGL_nvidia.so.0" 2>/dev/null | head -1)):$LD_LIBRARY_PATH
export NVIDIA_DRIVER_CAPABILITIES=all

python - <<'PY'
import os
os.environ["MAGNUM_LOG"] = "quiet"
os.environ["HABITAT_SIM_LOG"] = "quiet"
import habitat

CONFIG = "/content/habitat-lab-v033/habitat-lab/habitat/config/benchmark/nav/pointnav/pointnav_habitat_test.yaml"
config = habitat.get_config(config_path=CONFIG, overrides=[
    "habitat.environment.max_episode_steps=5",
    "habitat.environment.iterator_options.shuffle=False",
])

env = habitat.Env(config=config)
obs = env.reset()

print(f"Observation type: {type(obs).__name__}")
print(f"Obs keys: {list(obs.keys())}")

obs = env.step({"action": "move_forward"})
print(f"\nAfter step: still just observations")
print(f"Metrics (separate call): {env.get_metrics()}")
env.close()
PY

Observation type: Observations
Obs keys: ['rgb', 'depth', 'pointgoal_with_gps_compass']

After step: still just observations
Metrics (separate call): {'distance_to_goal': 6.554908752441406, 'success': 0.0, 'spl': 0.0, 'distance_to_goal_reward': -0.2197256088256836}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
pybullet build time: Jan 29 2025 23:20:52
PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring
2026-04-18 05:58:56,178 Initializing dataset PointNav-v1
2026-04-18 05:58:56,465 initializing sim Sim-v0
PluginManager::Manager: duplicate static plugin S

In [39]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv
python -c "
import habitat.gym as g
print(dir(g))
"


['__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'gym_definitions', 'gym_env_episode_count_wrapper', 'gym_env_obs_dict_wrapper', 'gym_wrapper', 'make_gym_from_config']


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
pybullet build time: Jan 29 2025 23:20:52
PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring


## 4. Gym-wrapped — GymHabitatEnv
Same task, but with standard Gym API (obs/reward/done/info tuple)

In [40]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv
cd /content && export MPLBACKEND=Agg
export __EGL_VENDOR_LIBRARY_FILENAMES=/tmp/nvidia_egl.json
export LD_LIBRARY_PATH=$(dirname $(find /usr -name "libEGL_nvidia.so.0" 2>/dev/null | head -1)):$LD_LIBRARY_PATH
export NVIDIA_DRIVER_CAPABILITIES=all

python - <<'PY'
import os
os.environ["MAGNUM_LOG"] = "quiet"
os.environ["HABITAT_SIM_LOG"] = "quiet"

import habitat
from habitat.gym import make_gym_from_config

CONFIG = "/content/habitat-lab-v033/habitat-lab/habitat/config/benchmark/nav/pointnav/pointnav_habitat_test.yaml"
config = habitat.get_config(config_path=CONFIG, overrides=[
    "habitat.environment.max_episode_steps=10",
    "habitat.environment.iterator_options.shuffle=False",
])

# Wrap as Gym env
env = make_gym_from_config(config=config)

print(f"Action space: {env.action_space}")
print(f"Observation space keys: {list(env.observation_space.spaces.keys())}")

obs = env.reset()
print(f"\n--- Running 5 random steps ---")
for step in range(5):
    action = env.action_space.sample()  # random action
    obs, reward, done, info = env.step(action)
    print(f"Step {step}: reward={reward:.3f}, done={done}, distance_to_goal={info.get('distance_to_goal', 'N/A')}")
    if done:
        break

env.close()
PY

Action space: Discrete(4)
Observation space keys: ['depth', 'pointgoal_with_gps_compass', 'rgb']

--- Running 5 random steps ---
Step 0: reward=-0.010, done=False, distance_to_goal=6.335183143615723
Step 1: reward=-0.010, done=False, distance_to_goal=6.335183143615723
Step 2: reward=-0.010, done=True, distance_to_goal=6.335183143615723


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
pybullet build time: Jan 29 2025 23:20:52
PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring
2026-04-18 06:01:53,992 Initializing dataset PointNav-v1
2026-04-18 06:01:54,275 initializing sim Sim-v0
PluginManager::Manager: duplicate static plugin S

## 5. Why use Gym API?

**Plug into existing RL libraries:**
```python
from stable_baselines3 import PPO
model = PPO("MultiInputPolicy", env, verbose=1)
model.learn(total_timesteps=10000)
```

Because `env` follows the Gym interface, you don't write training loops yourself — RL libraries expect this exact API.

**Also standard tools work:**
- `env.observation_space.sample()` — sample random observation
- `env.action_space.sample()` — sample random action  
- Automatic input/output validation

## 6. Custom reward with Gym wrapper
You can wrap any Gym env to modify the reward before it reaches your RL algorithm.

In [41]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv
cd /content && export MPLBACKEND=Agg
export __EGL_VENDOR_LIBRARY_FILENAMES=/tmp/nvidia_egl.json
export LD_LIBRARY_PATH=$(dirname $(find /usr -name "libEGL_nvidia.so.0" 2>/dev/null | head -1)):$LD_LIBRARY_PATH
export NVIDIA_DRIVER_CAPABILITIES=all

python - <<'PY'
import os
os.environ["MAGNUM_LOG"] = "quiet"
os.environ["HABITAT_SIM_LOG"] = "quiet"
import gym
import habitat
from habitat.gym import make_gym_from_config

class NegDistanceRewardWrapper(gym.Wrapper):
    """Override reward to be negative distance to goal."""
    def step(self, action):
        obs, reward, done, info = self.env.step(action)
        custom_reward = -info.get("distance_to_goal", 0)
        return obs, custom_reward, done, info

CONFIG = "/content/habitat-lab-v033/habitat-lab/habitat/config/benchmark/nav/pointnav/pointnav_habitat_test.yaml"
config = habitat.get_config(config_path=CONFIG, overrides=[
    "habitat.environment.max_episode_steps=10",
    "habitat.environment.iterator_options.shuffle=False",
])

env = make_gym_from_config(config=config)
env = NegDistanceRewardWrapper(env)

obs = env.reset()
for step in range(5):
    action = env.action_space.sample()
    obs, reward, done, info = env.step(action)
    print(f"Step {step}: custom_reward={reward:.3f}, distance={info.get('distance_to_goal', 0):.3f}")
    if done:
        break

env.close()
PY

Step 0: custom_reward=-6.335, distance=6.335
Step 1: custom_reward=-6.335, distance=6.335
Step 2: custom_reward=-6.335, distance=6.335
Step 3: custom_reward=-6.335, distance=6.335


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
pybullet build time: Jan 29 2025 23:20:52
PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring
2026-04-18 06:02:25,121 Initializing dataset PointNav-v1
2026-04-18 06:02:25,402 initializing sim Sim-v0
PluginManager::Manager: duplicate static plugin S